# Step 1 - Set up MLflow

In [1]:
!pip install mlflow

In [3]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("MLflow Quickstart")

<Experiment: artifact_location='/home/junspring/mlflow-jjb/mlruns/2', creation_time=1767009949031, experiment_id='2', last_update_time=1767009949031, lifecycle_stage='active', name='MLflow Quickstart', tags={}>

# Step 2 - Prepare training data

In [4]:
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

# Step 3 - Train a model with MLflow Autologging

In [5]:
import mlflow

# Enable autologging for scikit-learn
mlflow.sklearn.autolog()

# Just train the model normally
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

2025/12/29 21:06:14 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '369a769b81be40758b40ed07a61c5b4b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
/home/junspring/anaconda3/envs/mlflow-env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,8888
,solver,'lbfgs'
,max_iter,1000
,multi_class,'auto'


# Step 4 - View the Run in the MLflow UI

In [8]:
!mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000

Registry store URI not provided. Using backend store URI.
2025/12/29 21:06:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/29 21:06:40 INFO mlflow.store.db.utils: Updating database tables
2025/12/29 21:06:40 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 21:06:40 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/29 21:06:40 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 21:06:40 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/29 21:06:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/29 21:06:40 INFO mlflow.store.db.utils: Updating database tables
2025/12/29 21:06:40 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 21:06:40 INFO alembic.runtime.migration: Will assume non-transactional DDL.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --

http://localhost:5000

# Step 5 - Log a model and metadata manually

In [6]:
# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Train the model
    lr = LogisticRegression(**params)
    lr.fit(X_train, y_train)

    # Log the model
    model_info = mlflow.sklearn.log_model(sk_model=lr, name="iris_model")

    # Predict on the test set, compute and log the loss metric
    y_pred = lr.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Optional: Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

/home/junspring/anaconda3/envs/mlflow-env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


# Step 6 - Load the model back for inference.

In [7]:
# Load the model back for predictions as a generic Python Function model
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

predictions = loaded_model.predict(X_test)

iris_feature_names = datasets.load_iris().feature_names

result = pd.DataFrame(X_test, columns=iris_feature_names)
result["actual_class"] = y_test
result["predicted_class"] = predictions

result[:4]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,predicted_class
0,6.1,2.8,4.7,1.2,1,1
1,5.7,3.8,1.7,0.3,0,0
2,7.7,2.6,6.9,2.3,2,2
3,6.0,2.9,4.5,1.5,1,1
